In [1]:
# Observation data collection snd inverntory filtering
import sqlite3
import pandas as pd
from pathlib import Path
from IPython.display import HTML, display

def get_observation_data(start_date=None, end_date=None, db_path='nursery.db'):
    """
    Read observation data from the OBSERVACION_LOTE table with optional date filtering.
    """
    # Convert to Path object and resolve relative path
    db_file = Path(db_path)
    
    # Check if database exists
    if not db_file.exists():
        raise FileNotFoundError(f"Database file not found: {db_file.absolute()}")
    
    try:
        # Connect to database 
        conn = sqlite3.connect(str(db_file))

        # Build the query with optional date filtering
        query = """ 
        SELECT * FROM OBSERVACION_LOTE
        WHERE 1=1 -- Always true makes it easier to add conditions
        """

        # Add date filters if provided
        params = []
        if start_date:
            query += " AND FECHA_OBSERVACION >= ?"
            params.append(start_date)
        if end_date:
            query += " AND FECHA_OBSERVACION <= ?"
            params.append(end_date)

        # Order by date (most recent first)
        query += " ORDER BY FECHA_OBSERVACION DESC"

        # Execute the query and get data
        df = pd.read_sql_query(query, conn, params=params if params else None)
        
        return df
    
    except sqlite3.Error as e:
        print(f"Database error: {e}")
        raise
    finally:
        if 'conn' in locals():
            conn.close()

def generate_html_report(df, title="VIVERO RAÍCES DORADAS - REPORTE OPERATIVO"):
    """
    Generate an HTML report from observation data.
    
    Args:
        df (pandas.DataFrame): Observation data
        title (str): Report title
    
    Returns:
        str: HTML string
    """
    if df.empty:
        return "<h3>No data available</h3>"
    
    # Calculate statistics
    total_observations = len(df)
    date_range = f"{df['FECHA_OBSERVACION'].min()} a {df['FECHA_OBSERVACION'].max()}"
    unique_lots = df['ESPECIE_LOTE_ID'].nunique()
    available_columns = len(df.columns)
    
    # Create the HTML
    html_content = f'''
    <center><h1 style="color: #777777;">{title}</h1></center>
    <center><h2 style="color: #777777;">Observación de Plantas</h2></center>
    
    <div align="center">
        <table style="width: 80%; border-collapse: collapse; margin: 20px auto; font-family: Arial, sans-serif;">
            <tr>
                <th style="background-color: #e3d0b3; color: #926e41; padding: 12px; border: 1px solid #926e41;">Rango de Fechas</th>
                <th style="background-color: #e3d0b3; color: #926e41; padding: 12px; border: 1px solid #926e41;">Conteo de Observaciones</th>
                <th style="background-color: #e3d0b3; color: #926e41; padding: 12px; border: 1px solid #926e41;">Lotes Existentes</th>
                <th style="background-color: #e3d0b3; color: #926e41; padding: 12px; border: 1px solid #926e41;">Columnas disponibles</th>
            </tr>
            <tr>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #555;">{date_range}</td>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #555;">{total_observations:,}</td>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #555;">{unique_lots}</td>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #555;">{available_columns}</td>
            </tr>
        </table>
    </div>
    
    <div align="center" style="margin: 20px;">
        <h3 style="color: #777777;">Resumen histórico de datos</h3>
        <table style="width: 60%; border-collapse: collapse; margin: 10px auto; font-family: Arial, sans-serif;">
            <tr>
                <th style="background-color: #f5f5f5; color: #555; padding: 8px; border: 1px solid #ddd;">Nombre de Columna</th>
                <th style="background-color: #f5f5f5; color: #555; padding: 8px; border: 1px solid #ddd;">Valores Únicos</th>
                <th style="background-color: #f5f5f5; color: #555; padding: 8px; border: 1px solid #ddd;">Valores Totales</th>
            </tr>
    '''
    
    # Add row for each column
    for column in df.columns:
        unique_count = df[column].nunique()
        value_count = df[column].shape[0]
        
        html_content += f'''
            <tr>
                <td style="padding: 8px; border: 1px solid #ddd; color: #333;">{column}</td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: #666;">{unique_count}</td>
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: #666;">{value_count}</td>
            </tr>
        '''
    
    return html_content


def display_report(df=None, start_date=None, end_date=None, db_path='nursery.db'):
    """
    Complete function to get data and display HTML report.
    
    Args:
        df (pandas.DataFrame, optional): Pre-loaded data. If None, loads from database.
        start_date (str, optional): Start date for filtering
        end_date (str, optional): End date for filtering
        db_path (str, optional): Database path
    """
    from datetime import datetime
    
    # Get data if not provided
    if df is None:
        print("📊 Cargando datos de la base de datos...")
        df = get_observation_data(start_date=start_date, end_date=end_date, db_path=db_path)
        print(f"✅ Datos cargados: {len(df)} registros")
    
    # Generate and display HTML report
    html_report = generate_html_report(df)
    #display(HTML(html_report) )
    
    # Return the dataframe for further analysis
    return df

# Create the current_inventory dataframe:
def get_current_inventory(df):
    """ 
    Extract current inventory from observatin data. 
    Returns only observations from the most recent date for a complete snapshot.
    """
    # Find the most recent date in the dataset
    most_recent_date = df['FECHA_OBSERVACION'].max()
    # Filter to include only the most recent date
    current_inventory = df[df['FECHA_OBSERVACION'] == most_recent_date].copy()

    print(f"✅ Inventario actual creado: {current_inventory.shape[0]} observaciones únicas")
    print(f"   Lotes únicos: {current_inventory['ESPECIE_LOTE_ID'].nunique()}")
    print(f"   Etapas de crecimiento únicas: {current_inventory['ETAPA_CRECIMIENTO'].nunique()}")
    print(f"   Fecha del inventario: {most_recent_date}")
    
    return current_inventory

# Usage workflow
if __name__ == "__main__":
    #print("Iniciando reporte del vivero...\n")
    
    # Step 1: Display observation data
    print("Generando reporte filtrado...")
    observations = display_report(start_date='2025-10-27')

    # Step 2: Extract current inventory
    current_inventory = get_current_inventory(observations)

Generando reporte filtrado...
📊 Cargando datos de la base de datos...
✅ Datos cargados: 1371 registros
✅ Inventario actual creado: 50 observaciones únicas
   Lotes únicos: 32
   Etapas de crecimiento únicas: 7
   Fecha del inventario: 2026-01-13


In [2]:
# Especies data collection
import sqlite3
import pandas as pd
from pathlib import Path
from typing import Optional, List, Dict, Any
from IPython.display import HTML, display

def get_left_table_data(
        db_path: str = 'nursery.db',
        table_name: str = 'ESPECIE_LOTE',
        columns: Optional[List[str]] = None,
        left_join_key: str = ' ID',
        rename_key_for_join: Optional[str] = None
) -> pd.DataFrame:
    """
    Read data from a dimension table for left join operations.
    Provides traceability and displays shape overview in jupiter.

    Args:
        db_path: Path to SQLite database
        table_name: Name of the left table
        columns: Specific columns to select (None=all)
        left_join_key: Primary key column name in the left table
        rename_key_for_join: Optional new name for the key column

    Returns:
        Dataframe with left table data ready for merging
    """

    print(f"📊Paso 1: Cargando tabla {table_name} para el reporte, con identificador único {left_join_key}")

    db_file = Path(db_path)

    if not db_file.exists():
        raise FileNotFoundError(f"Database file not found: {db_file.absolute()}")
    try:
        conn = sqlite3.connect(str(db_file))

        # Get table info for validation
        table_info = pd.read_sql_query(f"PRAGMA table_info({table_name})", conn)

        available_columns = table_info['name'].to_list()

        # Validate key column exists
        if left_join_key not in available_columns:
            raise ValueError(f"Columna identificador '{left_join_key}' se está en la tabla {table_name}")
        
        if columns:
            # Validate all requested columns exists
            missing_cols = [col for col in columns if col not in available_columns]

            if missing_cols:
                raise ValueError(f"Columnas NO disponibles en la tabla '{table_name}': {missing_cols}")
            
            column_str = ', '.join(columns)
            selected_columns = columns
        else:
            column_str = '*'
            selected_columns = available_columns

        # Build and execute query
        query = f"""
        SELECT {column_str}
        FROM {table_name}
        """

        df = pd.read_sql_query(query,conn)

        # Rename key column if specified
        if rename_key_for_join and rename_key_for_join != left_join_key:
            df = df.rename(columns={left_join_key: rename_key_for_join})
            print(f" ♻️ Columna identificadora renombrada de '{left_join_key}' to '{rename_key_for_join}'")

        # Display shape overview
        print(f"✅ TABLA CARGADA EXITOSAMENTE")
        print(f" Filas cargadas: {df.shape[0]}, Columnas cargadas: {df.shape[1]}")
        print(f" Columnas selecionadas: {', '.join(df.columns.tolist())} ")

        return df
    
    except sqlite3.Error as e:
        print(f"🔴 Database error: {e}")
        raise
    finally:
        if 'conn' in locals():
            conn.close()

# Load left table (species data)
especie_df = get_left_table_data(
    db_path='nursery.db',
    table_name='ESPECIE_LOTE',
    columns=['ID', 'NOMBRE_COMUN', 'NOMBRE_CIENTIFICO', 'FAMILIA', 'CATEGORIA'],
    left_join_key='ID',
    rename_key_for_join='ESPECIE_LOTE_ID'
)

📊Paso 1: Cargando tabla ESPECIE_LOTE para el reporte, con identificador único ID
 ♻️ Columna identificadora renombrada de 'ID' to 'ESPECIE_LOTE_ID'
✅ TABLA CARGADA EXITOSAMENTE
 Filas cargadas: 37, Columnas cargadas: 5
 Columnas selecionadas: ESPECIE_LOTE_ID, NOMBRE_COMUN, NOMBRE_CIENTIFICO, FAMILIA, CATEGORIA 


In [3]:
# Inventory enrichment
def enrich_with_left_table(
    right_df: pd.DataFrame,
    left_df: pd.DataFrame,
    right_join_key: str,
    left_join_key: str = None,
    columns_to_enrich: Optional[List[str]] = None,
    suffix: str = '_dim'
) -> pd.DataFrame:
    """ 
    Enrich right table data with left table information using left join.
    To be executed after loading left table data.

    Args:
        right_df: DataFrame with observation data
        left_df: DataFrame with data from get_left_table_data()
        right_join_key: Foreign key column name in right_df
        left_join_key: Key column name in left_df (if different)
        columns_to_enrich: Specific columns from left_df to include (None = all exept join key)
        sufix: Suffix to add to duplicate column names from left table

    Returs:
        Enriched DataFrame
    """
    print(f"🔗 Paso 2: Cruce de tablas creando un reporte unificado")
    #print(f"   Right table shape: {right_df.shape}")
    #print(f"   Left table shape: {left_df.shape}")
    #print(f"   Join key - Right: {right_join_key}")
    #print(f"   Join key - Left: {left_join_key if left_join_key else 'ID'}")
    
    # Use default left join key if not specified
    if left_join_key is None:
        left_join_key = right_join_key
    
    # Validate join keys exist
    if right_join_key not in right_df.columns:
        raise ValueError(f"Join key '{right_join_key}' not found in right table columns: {right_df.columns.tolist()}")
    
    if left_join_key not in left_df.columns:
        raise ValueError(f"Join key '{left_join_key}' not found in left table columns: {left_df.columns.tolist()}")
    
    # Prepare left table data for merge
    merge_df = left_df.copy()
    
    # Select specific columns to enrich
    if columns_to_enrich:
        # Always include the join key
        columns_to_select = [left_join_key] + columns_to_enrich
        merge_df = merge_df[columns_to_select]
    
    # Identify overlapping columns (excluding join keys)
    overlapping_cols = set(right_df.columns) & set(merge_df.columns) - {right_join_key, left_join_key}
    if overlapping_cols:
        print(f"   ⚠️  Warning: Overlapping column names: {list(overlapping_cols)}")
        print(f"   Will use suffix '{suffix}' for left table columns")
    
    # Perform left join
    #print(f"\n   Merging with parameters:")
    #print(f"   - how: 'left'")
    #print(f"   - left_on: {right_join_key}")
    #print(f"   - right_on: {left_join_key}")
    #print(f"   - suffixes: ('', '{suffix}')")
    
    enriched_df = pd.merge(
        right_df,
        merge_df,
        left_on=right_join_key,
        right_on=left_join_key,
        how='left',
        suffixes=('', suffix)
    )
    
    # Display results
    print(f"\n✅ CRUCE EXITOSO")
    print(f"   Tabla final: {enriched_df.shape[0]} filas × {enriched_df.shape[1]} columnas")
    print(f"   Incluidas {enriched_df.shape[1] - right_df.shape[1]} nuevas columnas")
    
    # Show merge summary
    matched_count = enriched_df[left_join_key].notna().sum()
    unmatched_count = len(enriched_df) - matched_count
    
    #print(f"\n📊 MERGE SUMMARY:")
    #print(f"   Total rows: {len(enriched_df)}")
    #print(f"   Matched rows: {matched_count} ({matched_count/len(enriched_df)*100:.1f}%)")
    #print(f"   Unmatched rows: {unmatched_count} ({unmatched_count/len(enriched_df)*100:.1f}%)")
    
    # Display preview of enriched data
    #print(f"\n🎯 ENRICHED DATA PREVIEW:")
    #display(enriched_df.head().style.set_caption("First 5 rows of enriched data"))
    
    return enriched_df

enriched_inventory = enrich_with_left_table(
    right_df=current_inventory,
    left_df=especie_df,
    right_join_key='ESPECIE_LOTE_ID',
    left_join_key='ESPECIE_LOTE_ID',
    columns_to_enrich=['NOMBRE_COMUN', 'NOMBRE_CIENTIFICO', 'FAMILIA', 'CATEGORIA']
)

# Export the final inventory for excel manipulation
#enriched_inventory.to_csv('inventory_28_12_25.csv', encoding='UTF-8-SIG', index=False)


🔗 Paso 2: Cruce de tablas creando un reporte unificado

✅ CRUCE EXITOSO
   Tabla final: 50 filas × 12 columnas
   Incluidas 4 nuevas columnas


In [4]:
# Report header
def generate_about_us_header():
    """
    Generate an engaging 'About Us' header for the nursery report.
    Designed for Spanish-speaking audience including operational staff and potential investors.
    """
    
    html_header = '''
    <div align="center" style="background: linear-gradient(135deg, #e3d0b3 0%, #f5e9d5 100%); 
            padding: 30px 20px; margin-bottom: 30px; border-radius: 10px; 
            border-left: 5px solid #4a6b3d; border-right: 5px solid #4a6b3d;">
        
        <div style="max-width: 800px; margin: 0 auto;">
            <!-- Main Title -->
            <h1 style="color: #4a6b3d; margin-bottom: 15px; font-size: 28px; 
                    text-shadow: 1px 1px 2px rgba(0,0,0,0.1);">
                🌱 Vivero Raíces Doradas
            </h1>
            
            <!-- Tagline -->
            <h2 style="color: #926e41; font-size: 20px; margin-bottom: 25px; 
                    font-style: italic; border-bottom: 2px dashed #c9b18b; 
                    padding-bottom: 15px;">
                No vendemos plantas, cultivamos restauración ambiental verificable
            </h2>
            
            <!-- Main Description -->
            <div style="text-align: left; color: #555; line-height: 1.6; 
                    background-color: rgba(255, 255, 255, 0.7); 
                    padding: 20px; border-radius: 8px; margin-bottom: 20px;">
                
                <p style="margin-bottom: 15px; font-size: 16px;">
                    <strong>🏗️ Nuestra Misión Operativa:</strong><br>
                    Somos un vivero especializado en la <strong>propagación local</strong> de especies 
                    nativas, frutales, ornamentales y exóticas, destinadas para 
                    siembra en espacio público de <strong>Alameda del Río</strong> en Barraqnuilla. Nuestro enfoque 
                    combina conocimiento practico con seguimiento verificable basado en datos.
                </p>
                
                <p style="margin-bottom: 15px; font-size: 16px;">
                    <strong>📊 Para Nuestro Equipo:</strong><br>
                    Este reporte representa nuestro <strong>inventario actual y capacidad productiva</strong>. 
                    Cada número cuenta una historia de cuidado, dedicación y crecimiento medible. 
                    Tu trabajo diario es valorado y documentado.
                </p>
                
                <p style="margin-bottom: 15px; font-size: 16px;">
                    <strong>💼 Para Inversionistas y Colaboradores:</strong><br>
                    Este proyecto, iniciado en <strong>Mayo 2025</strong>, demuestra que la sostenibilidad 
                    ambiental puede ser <strong>medible, escalable y replicable</strong>. Cada aporte no solo 
                    financia plantas, sino un <strong>sistema completo</strong> que incluye:
                </p>
                
                <div style="display: flex; justify-content: space-between; flex-wrap: wrap; 
                        margin: 20px 0;">
                    <div style="flex: 1; min-width: 200px; margin: 10px; padding: 15px; 
                            background-color: #f0f7e9; border-radius: 6px; border-left: 4px solid #4a6b3d;">
                        <strong style="color: #4a6b3d;">🌳 Árboles Listos</strong><br>
                        <small>Especies adaptadas localmente</small>
                    </div>
                    <div style="flex: 1; min-width: 200px; margin: 10px; padding: 15px; 
                            background-color: #f0f7e9; border-radius: 6px; border-left: 4px solid #926e41;">
                        <strong style="color: #926e41;">📈 Seguimiento Verificable</strong><br>
                        <small>Datos transparentes, auditables y reportes periódicos</small>
                    </div>
                    <div style="flex: 1; min-width: 200px; margin: 10px; padding: 15px; 
                            background-color: #f0f7e9; border-radius: 6px; border-left: 4px solid #777777;">
                        <strong style="color: #777777;">🎓 Transferencia de Conocimiento</strong><br>
                        <small>Acompañamiento en la adecuación inicial de la siembra</small>
                    </div>
                </div>
                
                <p style="margin-top: 20px; font-size: 16px; color: #4a6b3d; font-weight: bold;">
                    <strong>🌊 Impacto Medible:</strong> Este proyecto contribuye a la restauración ambiental 
                    de la cuenca del <strong>Arroyo León</strong>, en el espacio urbano del <strong>Atlántico</strong> colombiano.
                </p>
            </div>
            
            <!-- Call to Action -->
            <div style="background-color: #4a6b3d; color: white; padding: 15px; 
                    border-radius: 6px; margin-top: 20px;">
                <p style="margin: 0; font-size: 14px;">
                    <strong>📞 Para mayor información sobre colaboraciones o inversión:</strong><br>
                    Contactar a: <span style="font-weight: bold; color: #e3d0b3;">3057388182 o 3332227918</span><br>
                    <small style="color: #c9b18b;">Proyecto sin ánimo de lucro • Todos los recursos se reinvierten en la operación.</small>
                </p>
            </div>
            
            <!-- Report Indicator -->
            <div style="margin-top: 25px; padding: 10px; border-top: 1px solid #c9b18b;">
                <p style="color: #777777; font-size: 14px; margin: 0;">
                    📋 <strong>REPORTE DE INVENTARIO OPERATIVO</strong> • Generado el ''' + pd.Timestamp.now().strftime('%d/%m/%Y') + '''
                </p>
            </div>
        </div>
    </div>
    '''
    
    return html_header

In [5]:
# Summary species summary overview
def generate_species_summary(df, title="ANÁLISIS DE INVENTARIO"):
    """
    Generate an HTML report of a standard inventory analysis
    
    Args:
        df (pandas.DataFrame): Enriched observation data
        title (str): Report title
    
    Returns:
        str: HTML string
    """
    
    # Check if all required columns exist
    required_columns = ['CANTIDAD_PLANTAS_VIVAS', 'CATEGORIA', 'FAMILIA', 
                       'NOMBRE_CIENTIFICO', 'NOMBRE_COMUN']
    
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        return f'''
    <div align="center">
        <h3 style="color: #777777;">⚠️ Datos incompletos para análisis de inventario</h3>
        <p style="color: #666;">Faltan columnas necesarias: {", ".join(missing_columns)}</p>
    </div>
        '''
    
    # Aggregate the data by species
    inventory_summary = df.groupby([
        'CATEGORIA', 'FAMILIA', 'NOMBRE_CIENTIFICO', 'NOMBRE_COMUN'
    ]).agg({
        'CANTIDAD_PLANTAS_VIVAS': 'sum'
    }).reset_index()
    
    # Sort by plant count descending for readability
    inventory_summary = inventory_summary.sort_values(
        'CANTIDAD_PLANTAS_VIVAS', ascending=False
    )
    
    # Calculate totals
    total_plants = inventory_summary['CANTIDAD_PLANTAS_VIVAS'].sum()
    total_species = inventory_summary['NOMBRE_COMUN'].nunique()
    total_families = inventory_summary['FAMILIA'].nunique()
    
    # Start HTML content
    html_content = f'''
<center><h2 style="color: #777777;">{title}</h2></center>

<div align="center">
    <table style="width: 90%; border-collapse: collapse; margin: 20px auto; font-family: Arial, sans-serif;">
        <tr>
            <th style="background-color: #e3d0b3; color: #926e41; padding: 12px; border: 1px solid #926e41;">Plantas Totales</th>
            <th style="background-color: #e3d0b3; color: #926e41; padding: 12px; border: 1px solid #926e41;">Especies Únicas</th>
            <th style="background-color: #e3d0b3; color: #926e41; padding: 12px; border: 1px solid #926e41;">Familias Únicas</th>
        </tr>
        <tr>
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #555;">{total_plants:,}</td>
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #555;">{total_species}</td>
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #555;">{total_families}</td>
        </tr>
    </table>
</div>
'''
    
    # Add growth stage summary
    if 'ETAPA_CRECIMIENTO' in df.columns and 'CANTIDAD_PLANTAS_VIVAS' in df.columns:
        # Get report date
        latest_date = df['FECHA_OBSERVACION'].max() if 'FECHA_OBSERVACION' in df.columns else 'N/A'
        
        # Calculate current plant count by stage
        current_plant_counts = df.groupby('ETAPA_CRECIMIENTO')['CANTIDAD_PLANTAS_VIVAS'].sum()
        current_plant_counts = current_plant_counts.sort_values(ascending=False)
        
        # Get total plants for percentage calculation
        total_current_plants = current_plant_counts.sum()
        
        # Add growth stage summary section
        html_content += f'''
<div align="center" style="margin: 30px;">
    <h3 style="color: #777777;">Resumen por Etapa de Crecimiento</h3>
    <table style="width: 30%; border-collapse: collapse; margin: 10px auto; font-family: Arial, sans-serif;">
        <tr>
            <th style="background-color: #d9e7cb; color: #4a6b3d; padding: 10px; border: 1px solid #4a6b3d;">Etapa de Crecimiento</th>
            <th style="background-color: #d9e7cb; color: #4a6b3d; padding: 10px; border: 1px solid #4a6b3d;">Inventario Actual</th>
            <th style="background-color: #d9e7cb; color: #4a6b3d; padding: 10px; border: 1px solid #4a6b3d;">Porcentaje</th>
        </tr>
'''
        
        # Add rows for each growth stage
        for stage, plant_count in current_plant_counts.items():
            percentage = (plant_count / total_current_plants * 100) if total_current_plants > 0 else 0
            
            html_content += f'''
        <tr>
            <td style="padding: 8px; border: 1px solid #ddd; color: #333;">{stage}</td>
            <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: #4a6b3d; font-weight: bold;">{plant_count:,}</td>
            <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: #666;">{percentage:.1f}%</td>
        </tr>
'''
        
        # Add summary row
        html_content += f'''
        <tr style="background-color: #f5f5f5;">
            <td style="padding: 10px; border: 1px solid #ddd; color: #333; font-weight: bold;">TOTAL</td>
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center; color: #4a6b3d; font-weight: bold;">{total_current_plants:,}</td>
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center; color: #666; font-weight: bold;">100.0%</td>
        </tr>
    </table>
'''
        
        # Add date note
        html_content += f'''
    <p style="color: #777777; font-size: 12px; margin-top: 10px;">
        <em>Inventario actual basado en la última observación (Fecha más reciente: {latest_date})</em>
    </p>
</div>
'''
    
    # Add species detail table
    html_content += f'''
<div align="center" style="margin: 30px;">
    <h3 style="color: #777777;">Conteo por Especie</h3>
    <table style="width: 95%; border-collapse: collapse; margin: 10px auto; font-family: Arial, sans-serif;">
        <tr>
            <th style="background-color: #d9e7cb; color: #4a6b3d; padding: 10px; border: 1px solid #4a6b3d; text-align: left;">Categoría</th>
            <th style="background-color: #d9e7cb; color: #4a6b3d; padding: 10px; border: 1px solid #4a6b3d; text-align: left;">Familia</th>
            <th style="background-color: #d9e7cb; color: #4a6b3d; padding: 10px; border: 1px solid #4a6b3d; text-align: left;">Nombre Científico</th>
            <th style="background-color: #d9e7cb; color: #4a6b3d; padding: 10px; border: 1px solid #4a6b3d; text-align: left;">Nombre Común</th>
            <th style="background-color: #d9e7cb; color: #4a6b3d; padding: 10px; border: 1px solid #4a6b3d; text-align: center;">Cantidad</th>
        </tr>
'''
    
    # Add rows for each species
    for _, row in inventory_summary.iterrows():
        html_content += f'''
        <tr>
            <td style="padding: 8px; border: 1px solid #ddd; color: #333;">{row['CATEGORIA']}</td>
            <td style="padding: 8px; border: 1px solid #ddd; color: #333;">{row['FAMILIA']}</td>
            <td style="padding: 8px; border: 1px solid #ddd; color: #333;"><i>{row['NOMBRE_CIENTIFICO']}</i></td>
            <td style="padding: 8px; border: 1px solid #ddd; color: #333; font-weight: bold;">{row['NOMBRE_COMUN']}</td>
            <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: #4a6b3d; font-weight: bold;">{row['CANTIDAD_PLANTAS_VIVAS']:,}</td>
        </tr>
'''
    
    # Add summary row
    html_content += f'''
        <tr style="background-color: #f5f5f5;">
            <td colspan="4" style="padding: 10px; border: 1px solid #ddd; color: #333; font-weight: bold; text-align: right;">TOTAL DE PLANTAS:</td>
            <td style="padding: 10px; border: 1px solid #ddd; text-align: center; color: #4a6b3d; font-weight: bold;">{total_plants:,}</td>
        </tr>
    </table>
</div>
'''

    # Add seed availability section
    html_content += f'''
<div align="center" style="margin: 30px;">
    <h3 style="color: #777777;">🌱 Disponibilidad de Semillas</h3>
    <p style="color: #666; font-size: 14px; margin-bottom: 20px;">
        <em>Información sobre semillas disponibles para propagación. Estos valores son aproximados 
        y no están incluidos en el conteo total de plantas.</em>
    </p>
    
    <table style="width: 60%; border-collapse: collapse; margin: 10px auto; font-family: Arial, sans-serif; font-size: 13px;">
        <tr>
            <th style="background-color: #e3d0b3; color: #926e41; padding: 10px; border: 1px solid #926e41; text-align: left;">Categoría</th>
            <th style="background-color: #e3d0b3; color: #926e41; padding: 10px; border: 1px solid #926e41; text-align: left;">Nombre Común</th>
            <th style="background-color: #e3d0b3; color: #926e41; padding: 10px; border: 1px solid #926e41; text-align: center;">Conteo Estimado</th>
        </tr>
        
        <!-- Native seeds -->
        <tr style="background-color: #f9f9f9;">
        </tr>
        <tr>
            <td style="padding: 8px; border: 1px solid #ddd; color: #333; font-weight: bold;">NATIVO</td>
            <td style="padding: 8px; border: 1px solid #ddd; color: #333;">Roble Rosado</td>
            <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: #4a6b3d; font-weight: bold;">200+</td>
        </tr>
        <tr>
            <td style="padding: 8px; border: 1px solid #ddd; color: #333; font-weight: bold;">NATIVO</td>
            <td style="padding: 8px; border: 1px solid #ddd; color: #333;">Roble Morado</td>
            <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: #4a6b3d; font-weight: bold;">30-50</td>
        </tr>
        <tr>
            <td style="padding: 8px; border: 1px solid #ddd; color: #333; font-weight: bold;">NATIVO</td>
            <td style="padding: 8px; border: 1px solid #ddd; color: #333;">Guácimo</td>
            <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: #4a6b3d; font-weight: bold;">200+</td>
        </tr>
        
        <!-- Exotic seeds -->
        <tr style="background-color: #f9f9f9;">
        </tr>
        <tr>
            <td style="padding: 8px; border: 1px solid #ddd; color: #333; font-weight: bold;">EXOTICO</td>
            <td style="padding: 8px; border: 1px solid #ddd; color: #333;">Lluvia de Oro</td>
            <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: #4a6b3d; font-weight: bold;">100+</td>
        </tr>
        <tr>
            <td style="padding: 8px; border: 1px solid #ddd; color: #333; font-weight: bold;">EXOTICO</td>
            <td style="padding: 8px; border: 1px solid #ddd; color: #333;">Acacia</td>
            <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: #4a6b3d; font-weight: bold;">50+</td>
        </tr>
        <tr>
            <td style="padding: 8px; border: 1px solid #ddd; color: #333; font-weight: bold;">EXOTICO</td>
            <td style="padding: 8px; border: 1px solid #ddd; color: #333;">Almendro</td>
            <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: #4a6b3d; font-weight: bold;">50+</td>
        </tr>
        
        <!-- Fruit seeds -->
        <tr style="background-color: #f9f9f9;">
        </tr>
        <tr>
            <td style="padding: 8px; border: 1px solid #ddd; color: #333; font-weight: bold;">FRUTAL</td>
            <td style="padding: 8px; border: 1px solid #ddd; color: #333;">Anón</td>
            <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: #4a6b3d; font-weight: bold;">10-30</td>
        </tr>
        
        <!-- Summary note -->
        <tr style="background-color: #f0f7e9;">
            <td colspan="3" style="padding: 12px; border: 1px solid #4a6b3d; color: #4a6b3d; font-size: 12px; text-align: center;">
                <em>📝 Nota: Las semillas están disponibles para proyectos de propagación y restauración. 
                Contactar para coordinar recolecta o donaciones.</em>
            </td>
        </tr>
    </table>
</div>
'''
    
    return html_content

In [6]:
# Category report overview
def generate_category_report(df, category_name):
    """
    Generate a detailed report for a specific plant category.
    
    Args:
        df (pandas.DataFrame): Enriched observation data
        category_name (str): Name of the category to filter (e.g., 'ORNAMENTAL')
    
    Returns:
        str: HTML string
    """
    
    # Filter data for the specified category
    if 'CATEGORIA' not in df.columns:
        return''' 
    <div align="center">
        <h3 style="color: #777777;">⚠️ Datos incompletos</h3>
        <p style="color: #666;">Columna CATEGORÍA no encontrada en los datos</p>
    </div>
        '''
    
    # Check if category exists
    available_categories = df['CATEGORIA'].unique()
    if category_name not in available_categories:
        return f'''
    <div align="center">
        <h3 style="color: #777777;">⚠️ Categoría no encontrada</h3>
        <p style="color: #666;">La categoría "{category_name}" no existe. Categorías disponibles: {", ".join(available_categories)}</p>
    </div>
        '''
    
    # Filter data
    category_data = df[df['CATEGORIA'] == category_name].copy()
    
    if len(category_data) == 0:
        return f'''
    <div align="center">
        <h3 style="color: #777777;">⚠️ Sin datos</h3>
        <p style="color: #666;">No hay datos disponibles para la categoría "{category_name}"</p>
    </div>
        '''
    
    # Calculate category statistics
    total_plants = category_data['CANTIDAD_PLANTAS_VIVAS'].sum()
    unique_species = category_data['NOMBRE_COMUN'].nunique()
    unique_families = category_data['FAMILIA'].nunique()
    
    # Check if height data is available
    has_height_data = 'ALTURA_PROMEDIO_CM' in category_data.columns
    
    # Create pivot table for growth stages
    if 'ETAPA_CRECIMIENTO' in category_data.columns:
        # Pivot plant counts
        pivot_table = category_data.pivot_table(
            index='NOMBRE_COMUN', # Rows: Plant names
            columns='ETAPA_CRECIMIENTO', # Columns: Growth stages
            values='CANTIDAD_PLANTAS_VIVAS', # Values: Plant count
            aggfunc='sum', # Operation: SUM
            fill_value=0 # If no data put 0
        ).reset_index()
        
        # Add total column
        pivot_table['TOTAL'] = pivot_table.iloc[:, 1:].sum(axis=1)
        pivot_table = pivot_table.sort_values('TOTAL', ascending=False)
        
        # Get average height per species if available
    if has_height_data:
        # EXCLUDE 'SEMBRADO' from height calculations
        # Filter out rows where ETAPA_CRECIMIENTO is 'SEMBRADO'
        nursery_data = category_data[category_data['ETAPA_CRECIMIENTO'] != 'SEMBRADO'].copy()
        
        if not nursery_data.empty:
            # Calculate average height per species (excluding 'SEMBRADO')
            height_by_species = nursery_data.groupby('NOMBRE_COMUN')['ALTURA_PROMEDIO_CM'].mean().round(1)
            
            # Map to pivot table
            pivot_table['ALTURA_PROMEDIO_CM'] = pivot_table['NOMBRE_COMUN'].map(height_by_species)
            
            # Optional: Add note about exclusion
            print(f"   📏 Alturas calculadas excluyendo etapa 'SEMBRADO'")
        else:
            # If all data is 'SEMBRADO', we can't calculate nursery heights
            pivot_table['ALTURA_PROMEDIO_CM'] = None
            print(f"   ⚠️  No hay datos de altura disponibles (solo etapa 'SEMBRADO')")
        # Get growth stages for columns
        growth_stages = [col for col in pivot_table.columns 
                        if col not in ['NOMBRE_COMUN', 'TOTAL', 'ALTURA_PROMEDIO_CM']]
        
        # Build the table
        table_html = f'''
        <table style="width: 100%; border-collapse: collapse; margin: 10px auto; font-family: Arial, sans-serif; font-size: 12px;">
            <tr>
                <th style="background-color: #e3d0b3; color: #926e41; padding: 8px; border: 1px solid #926e41; text-align: left;">Especie</th>
        '''
        
        # Add growth stage columns
        for stage in growth_stages:
            table_html += f'''
                <th style="background-color: #e3d0b3; color: #926e41; padding: 8px; border: 1px solid #926e41; text-align: center;">{stage}</th>
            '''
        
        # Add height column header if we have height data
        if has_height_data:
            table_html += '''
                <th style="background-color: #e3d0b3; color: #926e41; padding: 8px; border: 1px solid #926e41; text-align: center;">Altura Promedio<br><small>(cm)</small></th>
            '''
        
        table_html += '''
                <th style="background-color: #e3d0b3; color: #926e41; padding: 8px; border: 1px solid #926e41; text-align: center;">Total</th>
            </tr>
        '''
        
        # Add data rows
        for _, row in pivot_table.iterrows():
            table_html += f'''
            <tr>
                <td style="padding: 6px; border: 1px solid #ddd; color: #333; font-weight: bold;">{row['NOMBRE_COMUN']}</td>
            '''
            
            # Add plant counts for each growth stage
            for stage in growth_stages:
                count = row[stage]
                if count > 0:
                    table_html += f'''
                <td style="padding: 6px; border: 1px solid #ddd; text-align: center; color: #4a6b3d; font-weight: bold;">{int(count)}</td>
                    '''
                else:
                    table_html += '''
                <td style="padding: 6px; border: 1px solid #ddd; text-align: center; color: #999;">-</td>
                    '''
            
            # Add height column if available
            if has_height_data:
                height_value = row['ALTURA_PROMEDIO_CM']
                if pd.notna(height_value):
                    table_html += f'''
                <td style="padding: 6px; border: 1px solid #ddd; text-align: center; color: #926e41; font-weight: bold;">{height_value:.1f}</td>
                    '''
                else:
                    table_html += '''
                <td style="padding: 6px; border: 1px solid #ddd; text-align: center; color: #999;">-</td>
                    '''
            
            # Add total plants
            table_html += f'''
                <td style="padding: 6px; border: 1px solid #ddd; text-align: center; color: #926e41; font-weight: bold;">{int(row['TOTAL'])}</td>
            </tr>
            '''
        
        # Add total row
        column_totals = pivot_table[growth_stages].sum()
        grand_total = pivot_table['TOTAL'].sum()
        
        table_html += f'''
            <tr style="background-color: #f5f5f5;">
                <td style="padding: 8px; border: 1px solid #ddd; color: #333; font-weight: bold;">TOTAL</td>
        '''
        
        for stage in growth_stages:
            stage_total = column_totals[stage]
            table_html += f'''
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: #926e41; font-weight: bold;">{int(stage_total)}</td>
            '''
        
        # Add empty height column in total row if needed
        if has_height_data:
            table_html += '''
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: #999;">-</td>
            '''
        
        table_html += f'''
                <td style="padding: 8px; border: 1px solid #ddd; text-align: center; color: #926e41; font-weight: bold;">{int(grand_total)}</td>
            </tr>
        </table>
        '''
    else:
        table_html = '''
        <p style="color: #666; text-align: center;">No hay datos de etapa de crecimiento disponibles</p>
        '''
    
    # Build final HTML
    html_content = f'''
    <center><h2 style="color: #777777;">REPORTE DE CATEGORÍA: {category_name}</h2></center>
    
    <div align="center">
        <table style="width: 80%; border-collapse: collapse; margin: 20px auto; font-family: Arial, sans-serif;">
            <tr>
                <th style="background-color: #e3d0b3; color: #926e41; padding: 12px; border: 1px solid #926e41;">Plantas Totales</th>
                <th style="background-color: #e3d0b3; color: #926e41; padding: 12px; border: 1px solid #926e41;">Especies Únicas</th>
                <th style="background-color: #e3d0b3; color: #926e41; padding: 12px; border: 1px solid #926e41;">Familias Botánicas</th>
                <th style="background-color: #e3d0b3; color: #926e41; padding: 12px; border: 1px solid #926e41;">% del Inventario</th>
            </tr>
            <tr>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #555;">{total_plants:,}</td>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #555;">{unique_species}</td>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #555;">{unique_families}</td>
                <td style="padding: 10px; border: 1px solid #ddd; text-align: center; font-weight: bold; color: #555;">
                    {(total_plants / df["CANTIDAD_PLANTAS_VIVAS"].sum() * 100):.1f}%
                </td>
            </tr>
        </table>
    </div>
    
    <div align="center" style="margin: 30px;">
        <h3 style="color: #777777;">Distribución por Etapa de Crecimiento</h3>
        {table_html}
    </div>
    '''
    
    return html_content

In [7]:
# Display and export complete report
def export_complete_report(enriched_inventory, filename="reporte_inventario.html"):
    """
    Generate and export the complete report as an HTML file.
    This function creates a standalone HTML file with only the report content.
    """
    
    # Combine all HTML content
    full_html = f'''
<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Reporte de Inventario - Vivero Raíces Doradas</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 20px;
            background-color: #f9f9f9;
            color: #333;
        }}
        .container {{
            max-width: 1200px;
            margin: 0 auto;
            background: white;
            padding: 20px;
            border-radius: 10px;
            box-shadow: 0 0 10px rgba(0,0,0,0.1);
        }}
        .print-button {{
            background-color: #4a6b3d;
            color: white;
            border: none;
            padding: 10px 20px;
            border-radius: 5px;
            cursor: pointer;
            margin: 20px 0;
            float: right;
        }}
        .print-button:hover {{
            background-color: #3a5530;
        }}
        @media print {{
            .print-button {{ display: none; }}
            body {{ background: white; }}
        }}
    </style>
</head>
<body>
    <div class="container">
        <button class="print-button" onclick="window.print()">🖨️ Imprimir Reporte</button>
        
        {generate_about_us_header()}
        {generate_species_summary(enriched_inventory)}
        {generate_category_report(enriched_inventory, "EXOTICO")}
        {generate_category_report(enriched_inventory, "ORNAMENTAL")}
        {generate_category_report(enriched_inventory, "NATIVO")}
        {generate_category_report(enriched_inventory, "FRUTAL")}
        
        <div align="center" style="margin-top: 40px; padding: 20px; background-color: #f9f9f9; border-top: 2px solid #e3d0b3;">
            <p style="color: #777777; font-size: 12px;">
                 • Reporte generado automáticamente • Vivero Raíces Doradas •
            </p>
        </div>
    </div>
</body>
</html>
'''
    
    # Save to file
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(full_html)
    
    print(f"✅ Reporte exportado exitosamente a: {filename}")
    print(f"📄 Puedes abrir el archivo en cualquier navegador web")
    
    # Also display it in the notebook
    display(HTML(full_html))
    
    return filename

# Usage:
export_complete_report(enriched_inventory, "reporte_inventario_vivero.html")

   📏 Alturas calculadas excluyendo etapa 'SEMBRADO'
   📏 Alturas calculadas excluyendo etapa 'SEMBRADO'
   📏 Alturas calculadas excluyendo etapa 'SEMBRADO'
   📏 Alturas calculadas excluyendo etapa 'SEMBRADO'
✅ Reporte exportado exitosamente a: reporte_inventario_vivero.html
📄 Puedes abrir el archivo en cualquier navegador web


Plantas Totales,Especies Únicas,Familias Únicas
933,32,21
Etapa de Crecimiento,Inventario Actual,Porcentaje
BOLSA_PEQUEÑA,544,58.3%
BOLSA_GRANDE,178,19.1%
PRE-EMBOLSE,86,9.2%
CUARENTENA,72,7.7%
PLANTULA_ALMACIGO,25,2.7%
SEMBRADO,22,2.4%
GERMINACION,6,0.6%
TOTAL,933,100.0%


'reporte_inventario_vivero.html'